# Chapter 4: Discrete Behavior Cloning

This Colab follows the Chapter 4 manuscript from the multimodal regression trap through a trained, discrete SO-101 action policy. A GPU runtime is strongly recommended for the real backbone cells.

In [ ]:
# Colab setup: install the three public chapter packages from GitHub.
import subprocess
import sys

if 'google.colab' in sys.modules:
    organization = 'Large-Robotics-Models-From-Scratch'
    requirements = [
        f'lrm-ch02[data] @ git+https://github.com/{organization}/lrm-code-chapter-2.git@main',
        f'lrm-ch03 @ git+https://github.com/{organization}/lrm-code-chapter-3.git@main',
        f'lrm-ch04[data] @ git+https://github.com/{organization}/lrm-code-chapter-4.git@codex/fix-vla-backbone-loading',
    ]
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', *requirements],
        text=True, capture_output=True,
    )
    if result.returncode:
        detail = '\n'.join(
            part for part in (result.stdout, result.stderr) if part
        )
        raise RuntimeError(
            f'Chapter package installation failed:\n{detail}'
        )
    print('Installed Chapter 2, Chapter 3, and Chapter 4 packages.')

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

from ch04.constants import ACTION_BINS, ACTION_DIM, ACTION_HORIZON

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print(f'label grid: H={ACTION_HORIZON}, D={ACTION_DIM}, B={ACTION_BINS}')

## 4.2 The multimodal regression trap

The observation contains no clue about which of two equally valid expert modes was chosen. MSE therefore learns the conditional mean: zero, where the demonstrations have almost no density.

In [ ]:
from ch04.exercises import (make_bimodal_actions, train_gmm_baseline,
                           train_mse_baseline)
from ch04.diagnostics import plot_bimodal_comparison

observations, expert_actions = make_bimodal_actions()
mse_model, mse_history = train_mse_baseline()
mixture, gmm_history = train_gmm_baseline()
with torch.no_grad():
    collapsed = mse_model(torch.zeros(1, 1)).item()
    log_weights, means, sigmas = mixture(torch.zeros(1, 1))
print(f'MSE prediction: {collapsed:+.3f} (expert modes are -1 and +1)')
print('mixture means: ', means[0].tolist())
print('mixture weights:', log_weights.exp()[0].tolist())

# Figure 4.4: one regressor collapses into the valley, the mixture does not.
plot_bimodal_comparison(expert_actions.numpy(), collapsed, mixture=mixture)
plt.show()

## 4.3 Fit the action tokenizer

We compute normalization statistics from training episodes only, then fit per-joint q01/q99 limits in that normalized space. The helper reads state and action columns directly from Arrow, so the full fit does not decode the camera videos.

In [ ]:
from ch04 import ActionTokenizer
from ch04.data import (DEFAULT_DATASET_ID, collect_normalized_actions,
                         make_chunked_dataloaders)

train_loader, validation_loader, stats = make_chunked_dataloaders(
    DEFAULT_DATASET_ID, batch_size=4, validation_fraction=0.1)
# None uses every training frame through the fast Arrow action column.
# Set an integer only for a bounded loader-based smoke fit.
TOKENIZER_FIT_BATCHES = None
normalized_actions = collect_normalized_actions(
    train_loader, stats, max_batches=TOKENIZER_FIT_BATCHES)
tokenizer = ActionTokenizer.fit(normalized_actions)

example = normalized_actions[0]
bins = tokenizer.encode(example)
decoded = tokenizer.decode(bins)
print('bins:   ', bins)
print('AR action embedding ids:', bins, '(separate 256-entry table)')
print('max normalized round-trip error:', np.abs(decoded-example).max())
print('tokenizer fit batches:', TOKENIZER_FIT_BATCHES or 'all')
print('train episodes:', train_loader.dataset.episodes)
print('validation episodes:', validation_loader.dataset.episodes)

In [ ]:
# A chunk has H vector-valued action positions and H x D labels.
demo_grid = torch.arange(ACTION_HORIZON * ACTION_DIM).reshape(
    1, ACTION_HORIZON, ACTION_DIM)
print('target grid:', tuple(demo_grid.shape))
print('action positions:', ACTION_HORIZON)
print('first two vectors:', demo_grid[0, :2].tolist())

## 4.4 Build the three action heads

The factorized head is the one-shot baseline, the autoregressive head is the exact chain-rule model, and the bidirectional parallel head is the manuscript's one-pass path. Each experiment below starts from a fresh Chapter 3 backbone so training one design cannot improve the next design's starting point.

In [ ]:
from ch03 import VLABackbone
from ch04 import (AutoregressiveActionHead, FactorizedActionHead,
                  ParallelDecodeActionHead)


def build_action_head(name):
    backbone = VLABackbone().to(device)
    backbone.language_backbone.set_attn_implementation('eager')
    builders = {
        'factorized': lambda: FactorizedActionHead(),
        'autoregressive': lambda: AutoregressiveActionHead(backbone),
        'parallel': lambda: ParallelDecodeActionHead(backbone),
    }
    if name not in builders:
        raise ValueError(f'unknown action head: {name}')
    return backbone, builders[name]().to(device)


print('experiment order: factorized -> autoregressive -> parallel')

## 4.5 Shared training, visualization, and evaluation

Only the logits path varies across architectures. Factorized and parallel logits come directly from the observation; autoregressive training and marginal visualization use teacher forcing. Complete autoregressive samples and decoded chunks still use causal KV-cached generation, preserving the chain-rule model during evaluation.

In [ ]:
import itertools

from ch04.data import action_targets, prepare_batch
from ch04.decoding import (decode_action_chunk, evaluate_open_loop,
                             evaluation_mode, sample_action_grids)
from ch04.diagnostics import (plot_action_distribution,
                              plot_chunk_comparison,
                              plot_joint_support, temporal_jitter,
                              within_expert_support)
from ch04.losses import masked_token_cross_entropy
from ch04.train import action_head_logits, train_action_head


def run_head_experiment(name, steps=10, samples=64, eval_batches=1):
    backbone, head = build_action_head(name)
    history = train_action_head(
        head, backbone, train_loader, stats, tokenizer, device,
        total_steps=steps, warmup_steps=min(5, steps - 1),
        log_every=max(1, steps // 20), checkpoint_every=steps,
        checkpoint_dir=f'/content/ch04-checkpoints/{name}')

    batch = next(iter(validation_loader))
    model_inputs = prepare_batch(batch, stats, device, backbone)
    target_bins, token_pad = action_targets(
        batch, stats, tokenizer, device)
    with torch.no_grad(), evaluation_mode(backbone), evaluation_mode(head):
        logits = action_head_logits(
            head, backbone, model_inputs, target_bins)
        validation_ce = masked_token_cross_entropy(
            logits, target_bins, token_pad).item()

    sampled_grids = sample_action_grids(
        head, backbone, model_inputs, n_samples=samples)
    prediction = decode_action_chunk(
        head, backbone, model_inputs, tokenizer, stats,
        strategy='argmax').cpu()
    metrics = evaluate_open_loop(
        head, itertools.islice(validation_loader, eval_batches),
        tokenizer, stats, backbone, device)

    expert = torch.as_tensor(batch['action']).float()
    expert_pairs = target_bins[:, 0, [4, 5]].cpu().numpy()
    draws = sampled_grids[:, 0, [4, 5]].cpu().numpy()
    supported = within_expert_support(draws, expert_pairs, 8.0)
    jitter = np.mean([
        temporal_jitter(grid) for grid in sampled_grids.cpu().numpy()
    ])

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    axes[0].plot([row['step'] for row in history],
                 [row['loss'] for row in history], label='CE')
    axes[0].plot([row['step'] for row in history],
                 [row['entropy'] for row in history], label='entropy')
    axes[0].set(xlabel='step', title=f'{name}: training')
    axes[0].legend()
    probabilities = logits[0, 0, 4].softmax(-1).cpu().numpy()
    plot_action_distribution(probabilities, ax=axes[1])
    axes[1].set_title('timestep 0, control 4')
    plot_joint_support(
        expert_pairs, draws, supported, 8.0, ax=axes[2])
    axes[2].set_title(f'{name}: complete-grid draws')
    fig.tight_layout()
    plt.show()
    plot_chunk_comparison(
        prediction[0].numpy(), expert[0].numpy())
    plt.suptitle(f'{name}: held-out action chunk', y=1.01)
    plt.show()

    mae_std = metrics['mae_in_standard_deviations'].nanmean().item()
    print(f'{name}: validation CE={validation_ce:.3f}, '
          f'MAE/std={mae_std:.3f}, sampled jitter={jitter:.3f}')
    head.cpu(); backbone.cpu()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {
        'head': head, 'backbone': backbone, 'history': history,
        'validation_ce': validation_ce, 'mae_std': mae_std,
        'jitter': float(jitter), 'prediction': prediction,
        'logits': logits.cpu(), 'samples': sampled_grids.cpu(),
    }

### 4.5.1 Factorized head

This is the fastest baseline: one backbone pass and independent logits for all 96 cells.

In [ ]:
TRAIN_STEPS = 10  # use 20_000 for a full, comparable run
results = {}
results['factorized'] = run_head_experiment(
    'factorized', steps=TRAIN_STEPS)

### 4.5.2 Autoregressive head

Training remains one teacher-forced pass. Sampling and argmax decoding perform 96 causal steps with a KV cache, so this cell takes longer.

### Training the other two heads

`train_action_head` accepts any of the three heads; only the call
signature inside it differs, so the loop below is the whole change. Give
each head a **fresh** backbone: sharing one would let the second head
start from a trunk the first had already adapted.

From a terminal the same runs are one command:

```bash
ch04-train --head all --steps 20000 --batch-size 32
```

In [ ]:
from ch04.cli import build_action_head

TRAIN_EVERY_HEAD = False  # set True to reproduce the figure 4.9 comparison
comparison_heads = {'parallel': head}
if TRAIN_EVERY_HEAD:
    for name in ('factorized', 'autoregressive'):
        peer_backbone = VLABackbone().to(device)
        peer = build_action_head(name, peer_backbone).to(device)
        train_action_head(
            peer, peer_backbone, train_loader, stats, tokenizer, device,
            total_steps=TRAIN_STEPS, warmup_steps=min(5, TRAIN_STEPS - 1),
            log_every=1, validation_loader=validation_loader,
            checkpoint_dir=f'/content/ch04-checkpoints/{name}')
        comparison_heads[name] = peer
else:
    # Untrained peers still show the structural difference in figure 4.9.
    comparison_heads['factorized'] = build_action_head(
        'factorized', backbone).to(device).eval()
    comparison_heads['autoregressive'] = build_action_head(
        'autoregressive', backbone).to(device).eval()
print('heads to compare:', list(comparison_heads))

In [ ]:
results['autoregressive'] = run_head_experiment(
    'autoregressive', steps=TRAIN_STEPS, samples=32)

In [ ]:
from ch04.analysis import (collect_cell_softmaxes, expert_pairs_from_batch,
                           joint_mismatch_samples, mismatch_rates,
                           neighborhood_softmax_figure,
                           sampled_grids_by_head, set_seed)
from ch04.diagnostics import (plot_joint_mismatch_panels,
                              plot_temporal_traces)

SEED, ANCHOR_INDEX, N_NEIGHBORS = 0, 0, 32
BASE_JOINT, PAIR_DIMS, TIMESTEP = 0, (4, 5), 0
set_seed(SEED)

# Figure 4.8: the held-out softmax cluster around one anchor frame.
collected = collect_cell_softmaxes(
    head, backbone, validation_loader, stats, tokenizer, device,
    timestep=TIMESTEP, control=BASE_JOINT, max_batches=8)
neighborhood_softmax_figure(
    collected, anchor_index=ANCHOR_INDEX,
    n_neighbors=min(N_NEIGHBORS, collected['states'].shape[0]),
    checkpoint='in-memory', seed=SEED)
plt.show()

In [ ]:
from ch04.diagnostics import temporal_jitter

# Figure 4.9: joint mismatch, one panel per head, on the same observation.
expert_pairs = expert_pairs_from_batch(
    batch, stats, tokenizer, device, dims=PAIR_DIMS, timestep=TIMESTEP)
pairs = joint_mismatch_samples(
    comparison_heads, backbone, model_inputs,
    dims=PAIR_DIMS, timestep=TIMESTEP, n_samples=512)
plot_joint_mismatch_panels(pairs, expert_pairs, bin_range=(0, ACTION_BINS))
plt.show()
print('off-diagonal rate:',
      mismatch_rates(pairs, ACTION_BINS // 2, ACTION_BINS // 2))

# The same comparison along time: factorized jitter against the
# conditioned heads' smoother traces.
grids = sampled_grids_by_head(
    comparison_heads, backbone, model_inputs, n_samples=12)
plot_temporal_traces(grids, control=PAIR_DIMS[0])
plt.show()
for name, grid in grids.items():
    print(f'{name:>15} temporal jitter:', round(temporal_jitter(grid[0]), 2))

In [ ]:
results['parallel'] = run_head_experiment(
    'parallel', steps=TRAIN_STEPS)

## 4.6 Compare the three learned policies

The short default run checks the complete pipeline; its numbers are not a meaningful architecture ranking. Use the same dataset split, seed, and training-step count for all three before comparing them. The tokenizer returns normalized actions, and open-loop evaluation converts them back to the dataset's raw units.

In [ ]:
from ch04.analysis import decoded_chunk_stream, open_loop_episode_trace
from ch04.diagnostics import (plot_execution_schedules,
                              plot_open_loop_episode)
from ch04.execution import execution_schedules

# Figure 4.10: the three section 4.7.2 schedules over the same stream.
chunks = decoded_chunk_stream(
    head, backbone, validation_loader, tokenizer, stats, device,
    max_batches=8)
plot_execution_schedules(execution_schedules(chunks), control=0)
plt.show()

# Figure 4.11: one held-out episode, expert against decoded commands.
trace = open_loop_episode_trace(
    head, backbone, validation_loader, tokenizer, stats, device,
    max_batches=8)
plot_open_loop_episode(trace['predicted'], trace['expert'], trace['valid'])
plt.show()

In [ ]:
names = ['factorized', 'autoregressive', 'parallel']
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for axis, metric, title in zip(
        axes, ['validation_ce', 'mae_std', 'jitter'],
        ['Held-out CE', 'MAE / training std', 'Sampled jitter']):
    axis.bar(names, [results[name][metric] for name in names])
    axis.set_title(title)
    axis.tick_params(axis='x', rotation=20)
fig.tight_layout()
plt.show()
for name in names:
    result = results[name]
    print(f"{name:14s} CE={result['validation_ce']:.3f}  "
          f"MAE/std={result['mae_std']:.3f}  "
          f"jitter={result['jitter']:.3f}")

## Next experiments

- Raise `TRAIN_STEPS` equally for all three heads before interpreting their diagnostic differences.
- Increase `samples` for smoother joint-support estimates; autoregressive sampling is intentionally slower.
- Compare chunk-by-chunk execution with `TemporalEnsembler` on validation episodes.
- Keep physical deployment separate: dataset units and the simulator or robot control mode must be converted and safety-checked explicitly.